In [11]:
# ---------------------------------------------------------
# GENERACIÓN DE OOF_LGB CON HIPERPARÁMETROS OPTUNA
# ---------------------------------------------------------

# 1. Definir los parámetros exactos que obtuviste
lgb_params = {
    'objective': 'regression',      # Cambiado para permitir ensamble continuo
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.2030094386568696,
    'num_leaves': 33,
    'max_depth': 60,
    'min_child_samples': 436,
    'lambda_l1': 2.16909215709766,
    'lambda_l2': 8.219565285043481,
    'random_state': 42
}

# 2. Inicializar el vector Out-of-Fold
oof_lgb = np.zeros(len(df))

# 3. Esquema de validación cruzada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando generación de OOF para LightGBM...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X[trn_idx], y[trn_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    # Pesos balanceados para manejar el desequilibrio de clases
    weights_tr = compute_sample_weight('balanced', y_tr)

    # Crear Datasets
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=weights_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    # Entrenar modelo del Fold
    model_lgb = lgb.train(
        lgb_params, 
        dtrain, 
        num_boost_round=2000, # Aumentado, early_stopping se encarga
        valid_sets=[dval], 
        callbacks=[
            lgb.early_stopping(100), 
            lgb.log_evaluation(0)
        ]
    )
    
    # Guardar predicciones en el vector OOF
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    
    # Cálculo rápido de Kappa por fold para monitorear
    # (Usando umbrales estándar temporales solo para control)
    fold_kappa = cohen_kappa_score(y_val, np.round(np.clip(oof_lgb[val_idx], 0, 4)), weights='quadratic')
    print(f"Fold {fold+1} completado. Kappa (redondeo simple): {fold_kappa:.4f}")

print("\nGeneración de oof_lgb finalizada.")

Iniciando generación de OOF para LightGBM...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[433]	valid_0's rmse: 1.11157
Fold 1 completado. Kappa (redondeo simple): 0.3385
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[607]	valid_0's rmse: 1.10359
Fold 2 completado. Kappa (redondeo simple): 0.3730
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[958]	valid_0's rmse: 1.09823
Fold 3 completado. Kappa (redondeo simple): 0.3640
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[549]	valid_0's rmse: 1.12197
Fold 4 completado. Kappa (redondeo simple): 0.3361
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[756]	valid_0's rmse: 1.08875
Fold 5 completado. Kappa (redondeo simple): 0.3801

Generación de oof_lgb finalizada.


In [13]:
# ---------------------------------------------------------
# GENERACIÓN DE OOF_TABNET CON PARÁMETROS OPTIMIZADOS
# ---------------------------------------------------------

# 1. Definir los parámetros de arquitectura que obtuviste
tabnet_params = {
    'n_d': 16,
    'n_a': 48,
    'n_steps': 8,
    'gamma': 1.0168168076162083,
    'lambda_sparse': 3.475715163340009e-05,
    'cat_idxs': cat_idxs,
    'cat_dims': cat_dims,
    'cat_emb_dim': 1,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=2e-2),
    'scheduler_params': {"step_size": 10, "gamma": 0.9},
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'mask_type': 'entmax',
    'verbose': 0
}

# 2. Inicializar el vector Out-of-Fold
oof_tabnet = np.zeros(len(df))

# 3. Esquema de validación cruzada (usando la misma semilla que en LGBM)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando generación de OOF para TabNet...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X[trn_idx], y[trn_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    # Pesos balanceados
    weights_tr = compute_sample_weight('balanced', y_tr)

    # Inicializar Regresor
    model_tn = TabNetRegressor(**tabnet_params)

    # Entrenar
    model_tn.fit(
        X_train=X_tr, 
        y_train=y_tr.reshape(-1, 1),
        eval_set=[(X_val, y_val.reshape(-1, 1))],
        eval_metric=['rmse'],
        max_epochs=200,
        patience=25,
        batch_size=1024, 
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
        weights=weights_tr
    )
    
    # Guardar predicciones continuas
    oof_tabnet[val_idx] = model_tn.predict(X_val).flatten()
    
    # Control rápido
    fold_kappa = cohen_kappa_score(y_val, np.round(np.clip(oof_tabnet[val_idx], 0, 4)), weights='quadratic')
    print(f"Fold {fold+1} completado. Kappa (redondeo simple): {fold_kappa:.4f}")

print("\nGeneración de oof_tabnet finalizada.")

Iniciando generación de OOF para TabNet...

Early stopping occurred at epoch 48 with best_epoch = 23 and best_val_0_rmse = 1.16107


c:\Users\fchiavassa\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Fold 1 completado. Kappa (redondeo simple): 0.2394

Early stopping occurred at epoch 39 with best_epoch = 14 and best_val_0_rmse = 1.14396


c:\Users\fchiavassa\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Fold 2 completado. Kappa (redondeo simple): 0.2627

Early stopping occurred at epoch 45 with best_epoch = 20 and best_val_0_rmse = 1.16798


...

Fold 3 completado. Kappa (redondeo simple): 0.2391

Early stopping occurred at epoch 51 with best_epoch = 26 and best_val_0_rmse = 1.15791


c:\Users\fchiavassa\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Fold 4 completado. Kappa (redondeo simple): 0.2529

Early stopping occurred at epoch 34 with best_epoch = 9 and best_val_0_rmse = 1.16924


c:\Users\fchiavassa\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Fold 5 completado. Kappa (redondeo simple): 0.2621

Generación de oof_tabnet finalizada.


In [14]:
oof_tabnet

array([2.24396849, 2.32098246, 2.74193001, ..., 2.13552856, 2.74512601,
       2.95765185], shape=(14993,))

In [15]:
# ---------------------------------------------------------
# 6. ENSAMBLE Y RESULTADOS
# ---------------------------------------------------------

oof_ensemble = (oof_lgb + oof_tabnet) / 2

rounder = OptimizedRounder()
rounder.fit(oof_ensemble, y)
final_preds = rounder.predict(oof_ensemble)

print("\n" + "="*30)
print(f"KAPPA ENSAMBLE FINAL: {cohen_kappa_score(y, final_preds, weights='quadratic'):.4f}")
print(f"Umbrales óptimos: {rounder.coef_}")
print("="*30)


KAPPA ENSAMBLE FINAL: 0.4023
Umbrales óptimos: [0.4502774  1.84832946 2.36794325 2.82806905]
